## Dimension Tables

This notebook builds the dimension tables in the `dbo` schema based on findings from 01_raw_exploration. All data quality issues identified in the exploration step are handled here.

Schema structure:
- `raw` — source tables loaded as-is from Kaggle or imports through DBeaver
- `dbo` — cleaned dimension and fact tables

Tables built:
- `dbo.dim_country` — country dimension with region, subregion, income group and land area
- `dbo.dim_year` — year dimension

Add subregion to dim_country:
Subregion added as a new column and populated from raw.Country_mapping using alpha-3 as the join key.

Notes:
- Taiwan (TWN) and Western Sahara (ESH) had NULL region after the mapping update. Fixed manually based on UN geographic classification.
- AGG_ aggregate entities (EU, World, income groups) have no region/subregion — expected
- Land_area_km2 sourced from raw.Land_Area
- Known edge cases (not errors): GRL (Europe & Central Asia / North America), BMU (North America / Caribbean), DJI (Middle East & North Africa / Eastern and Southern Africa)
- GRL (Greenland) excluded from statistical analysis — 80% ice, near-zero forest coverage, unusual World Bank classification

In [22]:
-- Create dim_country
CREATE TABLE dbo.dim_country (
    country_code    VARCHAR(50)     NOT NULL PRIMARY KEY,
    country_name    NVARCHAR(100),
    region          NVARCHAR(100),
    income_group    NVARCHAR(50)
);

Msg 2714, Level 16, State 6, Line 2
There is already an object named 'dim_country' in the database.

Total execution time: 00:00:00.020

In [ ]:
-- Insert all countries that have a valid ISO3 code
INSERT INTO dbo.dim_country (country_code, country_name, region, income_group)
SELECT DISTINCT
    f.Code          AS country_code,
    f.Country       AS country_name,
    i.region,
    i.income_group
FROM raw.Forest_year f
LEFT JOIN raw.income_groups i ON i.country_code = f.Code
WHERE f.Code IS NOT NULL AND f.Code <> '';

(214 rows affected)

Total execution time: 00:00:00.110

In [ ]:
-- Insert entities that have no ISO3 code
-- CONCAT builds AGG_ prefix e.g. AGG_European_Union
INSERT INTO dbo.dim_country (country_code, country_name, region, income_group)
SELECT DISTINCT
    CONCAT('AGG_', REPLACE(f.Country, ' ', '_')) AS country_code,
    f.Country       AS country_name,
    ig.region,
    ig.income_group
FROM raw.Forest_year f
LEFT JOIN raw.income_groups ig ON ig.country_code = f.Code
WHERE f.Code IS NULL OR f.Code = '';

(7 rows affected)

Total execution time: 00:00:00.040

In [6]:
-- Create dim_year
CREATE TABLE dbo.dim_year (year INT NOT NULL PRIMARY KEY);

Commands completed successfully.

Total execution time: 00:00:00.032

In [ ]:
-- Create dim_year with all years present in Forest_year (the largest date range)
INSERT INTO dbo.dim_year (year)
SELECT DISTINCT Year
FROM raw.Forest_year
ORDER BY Year;

(120 rows affected)

Total execution time: 00:00:00.039

In [25]:
-- Add subregion attribute to dim_country
ALTER TABLE dbo.dim_country
ADD subregion NVARCHAR(100);

Commands completed successfully.

Total execution time: 00:00:00.026

In [ ]:
-- Populate subregion from Country_mapping
-- [sub-region] needs brackets because of the dash in the column name
UPDATE c
SET c.subregion = m.[sub-region]
FROM dbo.dim_country c
JOIN raw.Country_mapping m ON m.[alpha-3] = c.country_code;

(213 rows affected)

Total execution time: 00:00:00.262

In [12]:
-- Verify subregion was populated correctly
-- Expected: 213 rows with subregion NOT NULL
SELECT region, subregion, COUNT(*) AS country_count
FROM dbo.dim_country
WHERE subregion IS NOT NULL
GROUP BY region, subregion
ORDER BY region, subregion


(28 rows affected)

region                     | subregion                       | country_count
---------------------------+---------------------------------+--------------
East Asia & Pacific        | Australia and New Zealand       | 2            
East Asia & Pacific        | East Asia                       | 5            
East Asia & Pacific        | Eastern Asia                    | 1            
East Asia & Pacific        | Melanesia                       | 5            
East Asia & Pacific        | Micronesia                      | 7            
East Asia & Pacific        | Polynesia                       | 5            
East Asia & Pacific        | South and Southeast Asia        | 11           
Europe & Central Asia      | Eastern Europe                  | 10           
Europe & Central Asia      | North America                   | 1            
Europe & Central Asia      | Northern Europe                 | 12           
Europe & Central Asia      | Southern Europe            

In [1]:
-- Manually fix missing regions for 2 countries not covered by Country_mapping
-- ESH (Western Sahara) → Middle East & North Africa (WB classification)
-- TWN (Taiwan) → East Asia & Pacific (WB classification)
UPDATE dbo.dim_country
SET region = 'Middle East & North Africa'
WHERE country_code = 'ESH';

UPDATE dbo.dim_country
SET region = 'East Asia & Pacific'
WHERE country_code = 'TWN';

(1 row affected)
(1 row affected)

Total execution time: 00:00:00.206

In [ ]:
-- Confirm both regions are now correctly set
SELECT country_code, country_name, region, subregion
FROM dbo.dim_country
WHERE country_name IN ('Taiwan', 'Western Sahara')

(2 rows affected)

country_code | country_name   | region | subregion      
-------------+----------------+--------+----------------
ESH          | Western Sahara | Africa | Northern Africa
TWN          | Taiwan         | Asia   | Eastern Asia   
(2 rows)

Total execution time: 00:00:00.020

In [ ]:
-- Expected NULLs:
-- region NULL: AGG_ aggregates, GUF (French Guiana), TWN (Taiwan)
-- subregion NULL: same as above + GRL has unusual classification (Europe & Central Asia / North America)
-- income_group NULL: AGG_ aggregates, ESH, GUF, TWN
SELECT country_code, country_name, region, subregion
FROM dbo.dim_country
WHERE region IS NULL OR subregion IS NULL

(8 rows affected)

country_code                      | country_name                  | region | subregion
----------------------------------+-------------------------------+--------+----------
AGG_England                       | England                       | NULL   | NULL     
AGG_European_Union_(27)           | European Union (27)           | NULL   | NULL     
AGG_High-income_countries         | High-income countries         | NULL   | NULL     
AGG_Low-income_countries          | Low-income countries          | NULL   | NULL     
AGG_Lower-middle-income_countries | Lower-middle-income countries | NULL   | NULL     
AGG_Scotland                      | Scotland                      | NULL   | NULL     
AGG_Upper-middle-income_countries | Upper-middle-income countries | NULL   | NULL     
OWID_WRL                          | World                         | NULL   | NULL     
(8 rows)

Total execution time: 00:00:00.033

In [ ]:
-- Add land_area_km2 column to dim_country
ALTER TABLE dbo.dim_country
    ADD land_area_km2 FLOAT;

In [ ]:
-- Populate land_area_km2 from raw.Land_Area using 2021 values
UPDATE dc
SET dc.land_area_km2 = la.[2021]
FROM dbo.dim_country dc
JOIN raw.Land_Area la ON la.[Country Code] = dc.country_code;

In [ ]:
-- Populate world total (OWID_WRL) as sum of all countries
UPDATE dbo.dim_country
SET land_area_km2 = (
    SELECT SUM(land_area_km2)
    FROM dbo.dim_country
    WHERE country_code NOT LIKE 'AGG_%'
        AND country_code != 'OWID_WRL'
        AND land_area_km2 IS NOT NULL
)
WHERE country_code = 'OWID_WRL';

In [14]:
-- Verify land_area_km2 populated
SELECT
    COUNT(*)                        AS total_countries,
    COUNT(land_area_km2)            AS filled,
    COUNT(*) - COUNT(land_area_km2) AS nulls
FROM dbo.dim_country;

(1 row affected)

total_countries | filled | nulls
----------------+--------+------
221             | 211    | 10   
(1 row)

Total execution time: 00:00:00.021